# Exercise 01: FGSM Attack

**Goal:** Implement the Fast Gradient Sign Method (FGSM) and evaluate its effectiveness at different perturbation strengths.

**Estimated time:** ~20 minutes

**Reference:** Goodfellow et al. 2014 — *Explaining and Harnessing Adversarial Examples*

## Background

FGSM is the simplest adversarial attack. Given an image `x`, true label `y`, and a model with parameters `θ`, it computes:

```
x_adv = x + ε · sign( ∇_x L(θ, x, y) )
```

The key idea: move in the direction that **increases the loss** by a small amount `ε`. The `sign()` function normalizes the gradient so every pixel is perturbed by exactly `ε`.

**Steps:**
1. Enable gradient tracking on the input image
2. Forward pass through the model, compute cross-entropy loss
3. Backward pass to get `∇_x L`
4. Apply perturbation: `x_adv = x + ε * sign(∇_x L)`
5. Clip to valid pixel range

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# Load pretrained ResNet18, set to eval mode
model = torchvision.models.resnet18(pretrained=True)
model.eval()

# Load 100 CIFAR-10 test images (use torchvision)
# Normalize with ImageNet stats since we use pretrained weights
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
testset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False)
images, labels = next(iter(testloader))

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

def predict(model, x):
    with torch.no_grad():
        logits = model(x)
    return logits.argmax(dim=1)

# Evaluate clean accuracy
clean_preds = predict(model, images)
print(f"Clean accuracy: {(clean_preds == labels).float().mean():.2%}")

## TODO: Implement FGSM

Fill in the `fgsm_attack` function below.

**Hints:**
- Use `images.clone().detach().requires_grad_(True)` to create a leaf tensor with gradient tracking
- Use `nn.CrossEntropyLoss()` for the loss
- Call `.backward()` and access `.grad` on your leaf tensor
- Use `torch.clamp(x_adv, min_val, max_val)` to clip the result

In [ ]:
def fgsm_attack(model, images, labels, epsilon):
    """
    Implement FGSM attack.
    Args:
        model: pretrained classifier
        images: batch of images (N, C, H, W), requires_grad will be set
        labels: true labels (N,)
        epsilon: perturbation magnitude
    Returns:
        adversarial images (N, C, H, W), clipped to valid range
    """
    # TODO:
    # 1. Enable gradient tracking on images
    # 2. Forward pass + compute cross-entropy loss
    # 3. Backward pass to get gradient w.r.t. images
    # 4. Apply perturbation: x_adv = x + epsilon * sign(grad)
    # 5. Clip to valid range (use image min/max or fixed [-3, 3] for normalized images)
    # HINT: use images.clone().detach().requires_grad_(True) to create a leaf tensor
    pass

## Evaluate FGSM at Multiple Epsilon Values

Test your FGSM implementation at `epsilon` values: `[0.01, 0.05, 0.1, 0.3, 0.5]`.

For each epsilon:
1. Generate adversarial images
2. Compute adversarial accuracy
3. Print the result

In [ ]:
epsilons = [0.01, 0.05, 0.1, 0.3, 0.5]
adv_accuracies = []

print(f"{'Epsilon':>10}  {'Adv Accuracy':>14}")
print("-" * 28)
for eps in epsilons:
    # TODO: Generate adversarial images and evaluate accuracy
    # adv_images = fgsm_attack(model, images, labels, eps)
    # adv_preds = predict(model, adv_images)
    # acc = (adv_preds == labels).float().mean().item()
    # adv_accuracies.append(acc)
    # print(f"{eps:>10.3f}  {acc:>14.2%}")
    pass

## Plot Accuracy vs. Epsilon

Plot clean accuracy (horizontal dashed line) and adversarial accuracy vs. epsilon.

In [ ]:
# TODO: Plot accuracy vs epsilon
# plt.figure(figsize=(8, 5))
# plt.plot(epsilons, adv_accuracies, 'o-', color='red', label='Adversarial accuracy')
# plt.axhline(y=(clean_preds == labels).float().mean().item(), color='blue',
#             linestyle='--', label='Clean accuracy')
# plt.xlabel('Epsilon')
# plt.ylabel('Accuracy')
# plt.title('FGSM: Accuracy vs. Perturbation Strength')
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.show()
pass

## Visualize Adversarial Examples

For epsilon=0.1, show 5 examples in a 3-row grid:
- Row 1: Original image + true label
- Row 2: Perturbation (scaled by 10 for visibility)
- Row 3: Adversarial image + predicted label

**Hint:** To display normalized images, use `transforms.Normalize` inverse or simply clip to [0, 1] after rescaling.

In [ ]:
def denormalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """Reverse ImageNet normalization for display."""
    t = tensor.clone()
    for i in range(3):
        t[i] = t[i] * std[i] + mean[i]
    return torch.clamp(t, 0, 1)

# TODO: Generate adversarial images at epsilon=0.1 and visualize 5 examples
# eps_viz = 0.1
# adv_images_viz = fgsm_attack(model, images, labels, eps_viz)
# adv_preds_viz = predict(model, adv_images_viz)
#
# fig, axes = plt.subplots(3, 5, figsize=(15, 9))
# for i in range(5):
#     orig = denormalize(images[i])
#     adv  = denormalize(adv_images_viz[i])
#     pert = adv_images_viz[i] - images[i]
#     pert_display = (pert - pert.min()) / (pert.max() - pert.min() + 1e-8)
#     ...
# plt.tight_layout()
# plt.show()
pass